In [2]:
import ir_datasets

# Load the dataset
dataset = ir_datasets.load("cranfield")

# Convert generators to standard Python lists so we can slice and iterate easily
documents = list(dataset.docs_iter())
queries = list(dataset.queries_iter())
qrels = list(dataset.qrels_iter())

# 1. Look at the first 3 documents
print("--- DOCUMENTS ---")
for doc in documents[:3]:
    print(f"Doc ID: {doc.doc_id}")
    print(f"Title: {doc.title}")
    print(f"Text: {doc.text[:100]}...\n")

# 2. Look at the first 3 queries
print("--- QUERIES ---")
for query in queries[:3]:
    print(f"Query ID: {query.query_id}")
    print(f"Text: {query.text}\n")

--- DOCUMENTS ---
Doc ID: 1
Title: experimental investigation of the aerodynamics of a
wing in a slipstream .
Text: experimental investigation of the aerodynamics of a
wing in a slipstream .
  an experimental study o...

Doc ID: 2
Title: simple shear flow past a flat plate in an incompressible fluid of small
viscosity .
Text: simple shear flow past a flat plate in an incompressible fluid of small
viscosity .
in the study of ...

Doc ID: 3
Title: the boundary layer in simple shear flow past a flat plate .
Text: the boundary layer in simple shear flow past a flat plate .
the boundary-layer equations are present...

--- QUERIES ---
Query ID: 1
Text: what similarity laws must be obeyed when constructing aeroelastic models
of heated high speed aircraft .

Query ID: 2
Text: what are the structural and aeroelastic problems associated with flight
of high speed aircraft .

Query ID: 3
Text: what problems of heat conduction in composite slabs have been solved so
far .



In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from collections import defaultdict

# Ensure required NLTK tools are downloaded
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    """Tokenizes, lowercases, removes stopwords, and stems the text."""
    tokens = word_tokenize(text.lower())
    return [stemmer.stem(word) for word in tokens if word.isalnum() and word not in stop_words]

# 1. BUILD THE INVERTED INDEX (Chapters 1 & 2)
inverted_index = defaultdict(list)

print("Building index... (This takes a few seconds)")
for doc in documents:
    doc_id = int(doc.doc_id)
    # Combine title and text to ensure we capture all relevant words
    processed_words = preprocess_text(doc.title + " " + doc.text)
    
    # Use set() so we only record the document ID once per word (Standard Boolean Indexing)
    for word in set(processed_words):
        inverted_index[word].append(doc_id)

# Sort the postings lists (Required for Delta Compression)
for word in inverted_index:
    inverted_index[word].sort()

print(f"Index built successfully! Total unique terms: {len(inverted_index)}")

# 2. DELTA & VARIABLE BYTE COMPRESSION (Chapter 4)
def delta_encode(postings):
    """Converts absolute document IDs to gaps (deltas)."""
    if not postings: return []
    deltas = [postings[0]]
    for i in range(1, len(postings)):
        deltas.append(postings[i] - postings[i-1])
    return deltas

def vbyte_encode_number(n):
    """Encodes a single integer into bytes using V-Byte."""
    bytes_list = []
    while True:
        bytes_list.insert(0, n % 128)
        if n < 128:
            break
        n = n // 128
    # Set the continuation bit (1) on the final byte
    bytes_list[-1] += 128
    return bytes_list

def vbyte_encode_list(numbers):
    """Applies V-Byte encoding to a whole list of numbers."""
    encoded = []
    for n in numbers:
        encoded.extend(vbyte_encode_number(n))
    return encoded

# Test compression on a highly frequent word
test_word = stemmer.stem("aerodynamics")
if test_word in inverted_index:
    original_postings = inverted_index[test_word]
    deltas = delta_encode(original_postings)
    compressed_bytes = vbyte_encode_list(deltas)
    
    print(f"\n--- COMPRESSION STATS FOR '{test_word}' ---")
    print(f"Original Postings (first 5): {original_postings[:5]}")
    print(f"Delta Encoded (first 5): {deltas[:5]}")
    # Assuming standard 32-bit (4 byte) integers for standard storage
    print(f"Uncompressed Memory: {len(original_postings) * 4} bytes")
    print(f"V-Byte Compressed Memory: {len(compressed_bytes)} bytes")